# 02 — Da seleção fotométrica à validação espectroscópica

Workshop SAB — AI-Scope / ADSS

**Pergunta científica:** depois de selecionar candidatos em um catálogo, quais deles possuem informação espectroscópica independente?

O exemplo usa uma tabela local de alvos e faz o crossmatch no servidor com DESI DR1. Isso evita baixar um catálogo espectroscópico inteiro apenas para cruzá-lo localmente.
`upload.targets` é a tabela temporária enviada em `file=targets`. `cone` compara posições em graus; `dist` mede a separação angular para examinar a qualidade da associação.

### DESI — DESI DR1 / SpecZ
O **DESI** (Dark Energy Spectroscopic Instrument) é um espectrógrafo multiobjeto instalado no telescópio Mayall de 4 metros, no Arizona.

- **Instrumento:** 5.000 posicionadores de fibras ópticas e espectrógrafos com três braços (B, R e Z).
- **Dados:** espectros ópticos, redshifts e classificações espectrais.
- **DESI SpecZ:** catálogo com redshifts (`z`), incertezas (`zerr`), indicadores de qualidade (`zwarn`) e classificações (`spectype`).
- **DESI DR1:** https://data.desi.lbl.gov/doc/releases/dr1/


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import adss

BASE_URL = "https://ai-scope.cbpf.br/"

cl = adss.ADSSClient(base_url=BASE_URL)

In [ ]:
# Use a saída do notebook anterior ou substitua por sua própria lista de alvos.
# Exemplo mínimo para permitir execução independente:
targets = pd.read_csv("outputs/cluster_1.csv")

In [ ]:
targets

### Consulta de redshifts DESI

Utilizando **astroql**, associe os objetos da tabela `upload.targets` ao catálogo espectroscópico `desi.desi_specz_dr1`, buscando correspondências dentro de um raio de **0,0003° (~1,08")**.

A consulta deve retornar:
- **Identificação:** ID Gaia (`source_id`) e ID DESI (`id`).
- **Coordenadas:** RA e Dec dos dois catálogos.
- **Espectroscopia:** `z`, `zerr`, `zwarn`, `spectype`, `subtype` e `deltachi2`.
- **Separação angular:** distância entre os objetos em graus. Dica: utilize `dist(dist(u.ra, u.dec, d.ra, d.dec) AS separation_deg)` como uma coluna para calcular a separação angular.

**Dicas:**
- Utilize `JOIN` para associar as tabelas.
- Utilize `cone()` para encontrar correspondências espaciais.
- Utilize `dist()` para calcular a separação angular.
- Utilize aliases (`AS`) para diferenciar as colunas dos catálogos.

**Atenção:** mantenha os aliases (renomear as colunas) `input_source_id`, `input_ra`, `input_dec`, `desi_id`, `desi_ra`, `desi_dec` e `separation_deg`, pois serão utilizados na análise seguinte.

In [ ]:
metadata = cl.get_database_metadata()

table = metadata.get_table("desi", "desi_specz_dr1")
if table is not None:
    print(table.column_names())

In [ ]:
query = """"""  # Complete a consulta AstroQL do enunciado.
if not query:
    raise NotImplementedError("Complete a query;")

spec = cl.query_and_wait(
    query_text=query,
    mode="astroql",
    file=targets,
    table_name="targets123",
).data

spec.head()

In [ ]:
spec["separation_arcsec"] = spec.separation_deg * 3600
print(spec.spectype.value_counts(dropna=False))

fig, ax = plt.subplots(figsize=(6,4))
ax.hist(spec.separation_arcsec.dropna(), bins=30)
ax.set(xlabel="Crossmatch separation [arcsec]", ylabel="N")
plt.show()

### Espectros DESI dos objetos associados

O catálogo `desi.desi_specz_dr1` fornece a classificação e o redshift. Os arrays espectrais estão em `desi.desi_dr1`, ligados por `desi_id = targetid`. Consultamos apenas os IDs encontrados acima e incluímos as três câmeras (B, R e Z).

`wave_*` contém comprimentos de onda, `flux_*` o fluxo espectral e `ivar_*` a **variância inversa** em cada pixel. Um pixel com `ivar <= 0` não tem peso útil no gráfico. As bandas B/R/Z são braços do espectrógrafo, não filtros fotométricos.

Agora associe os IDs únicos de `upload.desitargets` a `desi.desi_dr1` por `desi_id = targetid`. Selecione `wave`, `flux` e `ivar` para B, R e Z, além do ID. O nome da tabela carregada no ADSS deve coincidir com `table_name="desitargets"`.


In [ ]:
metadata = cl.get_database_metadata()

table = metadata.get_table("desi", "desi_dr1")
if table is not None:
    print(table.column_names())

In [ ]:
# Fetch one spectral row per matched DESI target, including all three arms.
desi_targets = spec[["desi_id", "desi_ra", "desi_dec"]].dropna().drop_duplicates(subset="desi_id")

spectra_query = """"""  # Complete a consulta AstroQL do enunciado.

if not spectra_query:
    raise NotImplementedError("Complete spectra_query;")

spectra = cl.query_and_wait(
    query_text=spectra_query,
    mode="astroql",
    file=desi_targets,
    table_name="desitargets",
).data

print(f"Matched DESI IDs: {len(desi_targets)}; spectra returned: {len(spectra)}")


In [ ]:
# Um alvo DESI pode aparecer em mais de uma linha; associe um z por ID.
specz_by_id = (spec.assign(desi_id=spec["desi_id"].astype("string"))
               .sort_values("separation_deg")
               .drop_duplicates("desi_id")
               [["desi_id", "input_source_id", "z", "zerr", "zwarn",
                 "spectype", "subtype", "deltachi2"]].copy())
specz_by_id["z"] = pd.to_numeric(specz_by_id["z"], errors="coerce")
specz_by_id["zerr"] = pd.to_numeric(specz_by_id["zerr"], errors="coerce")
specz_by_id["zwarn"] = pd.to_numeric(specz_by_id["zwarn"], errors="coerce")
spectra_with_z = spectra.assign(desi_id=spectra["desi_id"].astype("string")).merge(
    specz_by_id, on="desi_id", how="left", validate="many_to_one"
)

# Cada painel mostra os três braços do espectro e o resultado do ajuste DESI.
arms = {"b": "tab:blue", "r": "tab:orange", "z": "tab:red"}
page_size = 12
plotted = 0
for start in range(0, len(spectra_with_z), page_size):
    page = spectra_with_z.iloc[start:start + page_size]
    fig, axes = plt.subplots(3, 4, figsize=(18, 10), squeeze=False)
    for ax, (_, row) in zip(axes.flat, page.iterrows()):
        n_arms = 0
        for arm, color in arms.items():
            values = [row[f"{quantity}_{arm}"] for quantity in ("wave", "flux", "ivar")]
            if any(value is None for value in values):
                continue
            wave, flux, ivar = (np.asarray(value, dtype=float) for value in values)
            if wave.ndim != 1 or wave.shape != flux.shape or wave.shape != ivar.shape:
                continue
            valid = np.isfinite(wave) & np.isfinite(flux) & np.isfinite(ivar) & (ivar > 0)
            if valid.any():
                ax.plot(wave[valid], flux[valid], color=color, lw=0.6, label=arm.upper())
                n_arms += 1
        z_label = f"z={row.z:+.2e}" if pd.notna(row.z) else "z indisponível"
        ax.set_title(f"DESI {row.desi_id}  |  {row.spectype}\n{z_label}, ZWARN={row.zwarn}",
                     fontsize=8)
        if n_arms:
            ax.legend(fontsize=7, ncol=3, loc="upper right")
            plotted += 1
        else:
            ax.text(0.5, 0.5, "Sem pixels válidos", ha="center", va="center",
                    transform=ax.transAxes)
    for ax in axes.flat[len(page):]:
        ax.set_visible(False)
    fig.supxlabel("Comprimento de onda observado [Å]")
    fig.supylabel("Fluxo [10⁻¹⁷ erg s⁻¹ cm⁻² Å⁻¹]")
    fig.tight_layout()
    plt.show()
print(f"Espectros desenhados: {plotted} de {len(spectra_with_z)} linhas retornadas")


### `specz` para candidatos estelares

O `z` em `desi.desi_specz_dr1` é o ajuste espectral do Redrock. Para objetos classificados `STAR`, ele fica perto de zero; na aproximação de baixa velocidade, `v ≈ c z` ajuda a enxergar diferenças de poucos km/s. A figura mostra `z` contra a distância angular ao centro de M2 e a distribuição de `v ≈ c z` para alvos com `SPECTYPE=STAR` e `ZWARN=0`. Cada alvo DESI entra uma vez no resumo, mesmo que haja linhas repetidas no crossmatch ou nos espectros.

Isso é uma **checagem adicional**, não uma confirmação automática de pertença ao aglomerado: o crossmatch, a classificação, a incerteza e a distribuição de velocidades precisam ser avaliados. `ZWARN=0` indica ausência de avisos conhecidos do ajuste, não precisão garantida. Para velocidades radiais estelares de melhor qualidade, a [DESI DR1 Milky Way Survey VAC](https://data.desi.lbl.gov/doc/releases/dr1/vac/mws/) fornece medidas próprias de RVSpecFit; a [página de DR1](https://data.desi.lbl.gov/doc/releases/dr1/) descreve `SPECTYPE`, `ZWARN` e os três braços espectrais.


In [ ]:
C_KM_S = 299792.458
star_z = specz_by_id.merge(
    targets[["source_id", "radius_deg"]].rename(columns={"source_id": "input_source_id"}),
    on="input_source_id", how="left", validate="many_to_one",
)
star_z["radius_deg"] = pd.to_numeric(star_z["radius_deg"], errors="coerce")
star_z["v_approx_km_s"] = C_KM_S * star_z["z"]
good = (star_z["spectype"].eq("STAR") & star_z["zwarn"].eq(0)
        & np.isfinite(star_z["z"]) & np.isfinite(star_z["radius_deg"]))
good_stars = star_z.loc[good].copy()
print(f"Alvos DESI únicos: {len(star_z)}; estrelas com ZWARN=0 e z finito: {len(good_stars)}")

if good_stars.empty:
    print("Nenhum z estelar de boa qualidade para mostrar.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    zerr = good_stars["zerr"].fillna(0).clip(lower=0)
    axes[0].errorbar(good_stars["radius_deg"], good_stars["z"], yerr=zerr,
                     fmt="o", ms=4, capsize=2, alpha=0.8)
    axes[0].axhline(0, color="0.5", ls="--", lw=1)
    axes[0].set(xlabel="Distância ao centro de M2 [graus]", ylabel="z DESI",
                title="Redshift dos candidatos estelares")
    velocity_axis = axes[0].secondary_yaxis(
        "right", functions=(lambda z: C_KM_S * z, lambda v: v / C_KM_S)
    )
    velocity_axis.set_ylabel("v ≈ cz [km/s]")

    axes[1].hist(good_stars["v_approx_km_s"], bins=min(10, max(3, len(good_stars))),
                 color="tab:blue", alpha=0.8)
    axes[1].axvline(good_stars["v_approx_km_s"].median(), color="black", ls="--",
                    label="mediana")
    axes[1].set(xlabel="v ≈ cz [km/s]", ylabel="Número de alvos",
                title="Distribuição de velocidades aproximadas")
    axes[1].legend()
    plt.tight_layout()
    plt.show()

display(star_z[["input_source_id", "desi_id", "spectype", "z", "zerr",
                "zwarn", "v_approx_km_s", "radius_deg"]].sort_values("z"))


### Investigação

Compare os espectros, `SPECTYPE`, `ZWARN` e `z` dos candidatos. Os objetos com velocidade discrepante podem ser contaminantes ou ter ajustes problemáticos; examine também posição, movimentos próprios e qualidade do espectro. O valor `v ≈ cz` é uma aproximação didática, não uma medida dedicada de velocidade radial estelar.
